# ShiftLog-Gym Notebook 2: SFT + GRPO Curriculum

Real staged training with TRL + Unsloth (fallback supported), plus both token prompts (`WANDB_API_KEY`, `HF_TOKEN`).


In [ ]:
import os
REPO_URL='https://github.com/Chirag0096/ShiftLog-Gym.git'
REPO_DIR='ShiftLog-Gym'
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}
!pip -q install -U pip
!pip -q install -e . pandas matplotlib seaborn datasets accelerate peft bitsandbytes transformers huggingface_hub wandb
!pip -q install trl llm-blender mergekit
!pip -q uninstall -y unsloth unsloth_zoo || true


In [ ]:
import json, re
from getpass import getpass
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from datasets import Dataset
from huggingface_hub import login
from shiftlog_gym.scenarios import PUBLIC_FAMILIES
from shiftlog_gym.simulator import ShiftLogSimulator
from shiftlog_gym.training import summarize_episode, summarize_baseline, write_episode_replays
from shiftlog_gym.trl_env import ShiftLogToolEnv, reward_total as reward_total_env, reward_success as reward_success_env, reward_recall as reward_recall_env, reward_memory_write as reward_memory_write_env, reward_memory_integrity as reward_memory_integrity_env, reward_efficiency as reward_efficiency_env, reward_hallucination as reward_hallucination_env, reward_noise_resistance as reward_noise_resistance_env, reward_handoff as reward_handoff_env
sns.set_theme(style='whitegrid')
OBS_ROOT=Path('observatory')
RUNS_DIR=OBS_ROOT/'training_runs'
EPISODES_DIR=OBS_ROOT/'episodes'
OUTPUTS_DIR=Path('outputs')
for p in [OBS_ROOT, RUNS_DIR, EPISODES_DIR, OUTPUTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)
wandb_key=os.environ.get('WANDB_API_KEY','').strip()
if not wandb_key:
    wandb_key=getpass('Enter WANDB_API_KEY (blank to disable W&B): ').strip()
WANDB_ENABLED=bool(wandb_key)
if WANDB_ENABLED:
    os.environ['WANDB_API_KEY']=wandb_key
    import wandb
    wandb.login(key=wandb_key)
else:
    print('W&B disabled')
hf_token=os.environ.get('HF_TOKEN','').strip()
if not hf_token:
    hf_token=getpass('Enter HF_TOKEN (blank to skip HF login): ').strip()
if hf_token:
    os.environ['HF_TOKEN']=hf_token
    login(token=hf_token)
    print('HF login successful')
else:
    print('HF login skipped')


In [ ]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from trl import SFTConfig, SFTTrainer, GRPOConfig, GRPOTrainer

# Stable precision policy: choose exactly one mixed precision mode.
# Default to fp16 on Colab T4/L4; allow bf16 only if explicitly requested.
REQUEST_BF16 = os.environ.get('SHIFTLOG_USE_BF16', '0') == '1'
SUPPORTS_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
USE_BF16 = REQUEST_BF16 and SUPPORTS_BF16
USE_FP16 = not USE_BF16
os.environ['ACCELERATE_MIXED_PRECISION'] = 'bf16' if USE_BF16 else 'fp16'
print('Precision mode:', 'bf16' if USE_BF16 else 'fp16', '| bf16_supported=', SUPPORTS_BF16)

# IMPORTANT: keep Unsloth GRPO path off for stability.
USE_UNSLOTH_FOR_GRPO = False

MODEL_NAME = os.environ.get('SHIFTLOG_MODEL', 'Qwen/Qwen2.5-1.5B-Instruct')
FALLBACK_MODEL = os.environ.get('SHIFTLOG_FALLBACK_MODEL', 'Qwen/Qwen2.5-1.5B-Instruct')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = os.environ.get('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')

def load_model():
    model_id = MODEL_NAME or FALLBACK_MODEL
    compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_cfg,
        device_map='auto',
        torch_dtype=compute_dtype,
    )

    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)

    peft_cfg = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    )
    model = get_peft_model(model, peft_cfg)
    model.print_trainable_parameters()
    return model, tokenizer, '4bit+lora-stable'

model, tokenizer, backend = load_model()
print('Backend:', backend)
print('SFTTrainer module:', SFTTrainer.__module__)
print('GRPOTrainer module:', GRPOTrainer.__module__)


In [ ]:
RUN_STAGE_A=True
RUN_STAGE_B=True
RUN_STAGE_C=True
STAGE_B_FAMILIES=('db_pool','auth_cascade','oom_regression')
STAGE_C_FAMILIES=tuple(PUBLIC_FAMILIES)
def build_stage_dataset(families, steps, seed_offset):
    rows=[]
    for s in range(steps):
        fam=families[s % len(families)]
        var=s % 6
        seed=seed_offset+s+1
        rows.append({'prompt':[{'role':'user','content':f'Handle on-call incident for family={fam}, variant={var}, seed={seed}.'}], 'family':fam, 'variant_index':var, 'seed':seed})
    return Dataset.from_list(rows)
def parse_action(text, service_hint):
    m=re.search(r'\{.*\}', text, re.DOTALL)
    if not m: return {'tool':'inspect_service','arguments':{'service':service_hint}}
    try: obj=json.loads(m.group(0))
    except Exception: return {'tool':'inspect_service','arguments':{'service':service_hint}}
    if not isinstance(obj, dict) or 'tool' not in obj: return {'tool':'inspect_service','arguments':{'service':service_hint}}
    obj.setdefault('arguments',{})
    return obj
def rollout_eval(tag, model_obj, tok, families, episodes=20, max_steps=18):
    rows=[]; replay=[]
    for i in range(episodes):
        fam=families[i % len(families)]
        sim=ShiftLogSimulator()
        sim.reset(seed=8000+i, family=fam, variant_index=7)
        for _ in range(max_steps):
            if sim.done: break
            inc=sim.active_incident
            if inc is None: break
            prompt=sim.last_observation + '\nReturn one JSON tool call {"tool":...,"arguments":...}.'
            inputs=tok(prompt, return_tensors='pt').to(model_obj.device)
            out=model_obj.generate(**inputs, max_new_tokens=96, do_sample=False)
            text=tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            action=parse_action(text, inc.service)
            tool=action.get('tool','inspect_service'); args=action.get('arguments',{})
            if not hasattr(sim, tool):
                tool='inspect_service'; args={'service':inc.service}
            try: getattr(sim, tool)(**args)
            except Exception: sim.inspect_service(inc.service)
        art=summarize_episode(sim, f'{tag}-{fam}-{i:03d}', 'eval', 8000+i, 7)
        rows.append(art.episode_row); replay.append(art)
    write_episode_replays(EPISODES_DIR, replay)
    df=pd.DataFrame(rows)
    return df, summarize_baseline(rows)


def _resolve_envs(*args, **kwargs):
    if 'environments' in kwargs and kwargs['environments'] is not None:
        return kwargs['environments']
    if 'envs' in kwargs and kwargs['envs'] is not None:
        return kwargs['envs']
    if args and isinstance(args[0], list):
        return args[0]
    return None

def _fallback_zeros(**kwargs):
    completions = kwargs.get('completions') or []
    return [0.0] * len(completions)

def reward_total_safe(*args, **kwargs):
    envs = _resolve_envs(*args, **kwargs)
    return reward_total_env(envs, **kwargs) if envs is not None else _fallback_zeros(**kwargs)

def reward_success_safe(*args, **kwargs):
    envs = _resolve_envs(*args, **kwargs)
    return reward_success_env(envs, **kwargs) if envs is not None else _fallback_zeros(**kwargs)

def reward_recall_safe(*args, **kwargs):
    envs = _resolve_envs(*args, **kwargs)
    return reward_recall_env(envs, **kwargs) if envs is not None else _fallback_zeros(**kwargs)

def reward_memory_write_safe(*args, **kwargs):
    envs = _resolve_envs(*args, **kwargs)
    return reward_memory_write_env(envs, **kwargs) if envs is not None else _fallback_zeros(**kwargs)

def reward_memory_integrity_safe(*args, **kwargs):
    envs = _resolve_envs(*args, **kwargs)
    return reward_memory_integrity_env(envs, **kwargs) if envs is not None else _fallback_zeros(**kwargs)

def reward_efficiency_safe(*args, **kwargs):
    envs = _resolve_envs(*args, **kwargs)
    return reward_efficiency_env(envs, **kwargs) if envs is not None else _fallback_zeros(**kwargs)

def reward_hallucination_safe(*args, **kwargs):
    envs = _resolve_envs(*args, **kwargs)
    return reward_hallucination_env(envs, **kwargs) if envs is not None else _fallback_zeros(**kwargs)

def reward_noise_resistance_safe(*args, **kwargs):
    envs = _resolve_envs(*args, **kwargs)
    return reward_noise_resistance_env(envs, **kwargs) if envs is not None else _fallback_zeros(**kwargs)

def reward_handoff_safe(*args, **kwargs):
    envs = _resolve_envs(*args, **kwargs)
    return reward_handoff_env(envs, **kwargs) if envs is not None else _fallback_zeros(**kwargs)

def save_curve(log_history, stage_name):
    rows=[]
    for item in log_history:
        if 'step' not in item: continue
        rows.append({'step':item.get('step',0), 'reward_total':item.get('reward_total', item.get('reward',0.0)), 'reward_recall':item.get('reward_recall',0.0), 'reward_success':item.get('reward_success',0.0), 'reward_memory_write':item.get('reward_memory_write',0.0), 'reward_memory_integrity':item.get('reward_memory_integrity',0.0), 'recall_before_action_rate':item.get('recall_before_action_rate',0.0)})
    if not rows: rows=[{'step':0,'reward_total':0.0,'reward_recall':0.0,'reward_success':0.0,'reward_memory_write':0.0,'reward_memory_integrity':0.0,'recall_before_action_rate':0.0}]
    df=pd.DataFrame(rows).sort_values('step')
    csv_path=RUNS_DIR/f'training_curves_{stage_name}.csv'
    df.to_csv(csv_path, index=False)
    fig,ax=plt.subplots(figsize=(10,4))
    for col in ['reward_total','reward_recall','reward_success','reward_memory_write','recall_before_action_rate']:
        ax.plot(df['step'], df[col], label=col)
    ax.set_title(f'{stage_name.upper()} curves'); ax.set_xlabel('step'); ax.set_ylabel('score'); ax.legend(loc='best')
    fig.tight_layout(); fig.savefig(RUNS_DIR/f'training_curves_{stage_name}.png', dpi=180); plt.show()
    return csv_path


## Stage A: optional SFT bootstrap (50 steps)


In [ ]:
if RUN_STAGE_A:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    examples=[]
    for i in range(50):
        fam=PUBLIC_FAMILIES[i % len(PUBLIC_FAMILIES)]
        prompt=f'Family={fam}, provide first safe tool-call JSON.'
        completion=json.dumps({'tool':'read_shift_log','arguments':{'query':fam,'limit':3}})
        examples.append({'text': prompt + "\n" + completion})

    sft_ds=Dataset.from_list(examples)

    sft_cfg=SFTConfig(
        output_dir='outputs/stage-a-sft',
        max_steps=50,
        learning_rate=5e-6,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        logging_steps=5,
        save_steps=25,
        report_to=['wandb'] if WANDB_ENABLED else [],
        completion_only_loss=False,
        dataset_text_field='text',
        max_length=256,
        packing=False,
        gradient_checkpointing=True,
        optim='paged_adamw_8bit',
        fp16=USE_FP16,
        bf16=USE_BF16,
    )

    try:
        trainer_a=SFTTrainer(
            model=model,
            args=sft_cfg,
            train_dataset=sft_ds,
            processing_class=tokenizer,
        )
        trainer_a.train()
        trainer_a.save_model('outputs/stage-a-sft')
        tokenizer.save_pretrained('outputs/stage-a-sft')
        print('Stage A trainer:', type(trainer_a).__name__)
        print('Stage A curve:', save_curve(trainer_a.state.log_history, 'stageA'))
    except Exception as exc:
        print(f'SFTTrainer attempt failed ({type(exc).__name__}), switching to transformers Trainer fallback...')
        tokenized = sft_ds.map(lambda x: tokenizer(x['text'], truncation=True, max_length=192), remove_columns=['text'])
        collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
        args = TrainingArguments(
            output_dir='outputs/stage-a-sft',
            max_steps=50,
            learning_rate=5e-6,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            logging_steps=5,
            save_steps=25,
            report_to=['wandb'] if WANDB_ENABLED else [],
            remove_unused_columns=False,
            gradient_checkpointing=True,
            optim='paged_adamw_8bit',
            fp16=USE_FP16,
            bf16=USE_BF16,
        )
        trainer_a = Trainer(
            model=model,
            args=args,
            train_dataset=tokenized,
            tokenizer=tokenizer,
            data_collator=collator,
        )
        trainer_a.train()
        trainer_a.save_model('outputs/stage-a-sft')
        tokenizer.save_pretrained('outputs/stage-a-sft')
        print('Stage A fallback curve:', save_curve(trainer_a.state.log_history, 'stageA'))
else:
    print('Stage A skipped')


## Stage B: GRPO short rollout (200 steps)


In [ ]:
if RUN_STAGE_B:
    ds_b=build_stage_dataset(STAGE_B_FAMILIES, 200, 1000)
    cfg_b=GRPOConfig(output_dir='outputs/grpo-stage-b', max_steps=200, learning_rate=5e-6, per_device_train_batch_size=1, gradient_accumulation_steps=4, num_generations=2, max_completion_length=384, logging_steps=10, save_steps=50, report_to=['wandb'] if WANDB_ENABLED else [], log_completions=True, gradient_checkpointing=True, fp16=USE_FP16, bf16=USE_BF16)
    trainer_b=GRPOTrainer(model=model, args=cfg_b, train_dataset=ds_b, processing_class=tokenizer, environment_factory=lambda **kwargs: ShiftLogToolEnv(rollout_mode='short', multi_shift=False), reward_funcs=[reward_total_safe,reward_success_safe,reward_recall_safe,reward_memory_write_safe,reward_memory_integrity_safe,reward_efficiency_safe,reward_hallucination_safe,reward_noise_resistance_safe,reward_handoff_safe])
    trainer_b.train()
    trainer_b.save_model('outputs/grpo-stage-b')
    tokenizer.save_pretrained('outputs/grpo-stage-b')
    print('Stage B curve:', save_curve(trainer_b.state.log_history, 'stageB'))
else:
    print('Stage B skipped')


## Stage C: GRPO full rollout (300 steps)


In [ ]:
if RUN_STAGE_C:
    ds_c=build_stage_dataset(STAGE_C_FAMILIES, 300, 2000)
    cfg_c=GRPOConfig(output_dir='outputs/grpo-stage-c', max_steps=300, learning_rate=5e-6, per_device_train_batch_size=1, gradient_accumulation_steps=4, num_generations=2, max_completion_length=768, logging_steps=10, save_steps=50, report_to=['wandb'] if WANDB_ENABLED else [], log_completions=True, gradient_checkpointing=True, fp16=USE_FP16, bf16=USE_BF16)
    trainer_c=GRPOTrainer(model=model, args=cfg_c, train_dataset=ds_c, processing_class=tokenizer, environment_factory=lambda **kwargs: ShiftLogToolEnv(rollout_mode='full', multi_shift=False), reward_funcs=[reward_total_safe,reward_success_safe,reward_recall_safe,reward_memory_write_safe,reward_memory_integrity_safe,reward_efficiency_safe,reward_hallucination_safe,reward_noise_resistance_safe,reward_handoff_safe])
    trainer_c.train()
    trainer_c.save_model('outputs/grpo-stage-c')
    tokenizer.save_pretrained('outputs/grpo-stage-c')
    print('Stage C curve:', save_curve(trainer_c.state.log_history, 'stageC'))
else:
    print('Stage C skipped')


## Stage-wise eval and artifact integrity checks


In [ ]:
summaries={}
for stage_name, fams in [('stageA', STAGE_B_FAMILIES), ('stageB', STAGE_B_FAMILIES), ('stageC', STAGE_C_FAMILIES)]:
    df, summary=rollout_eval(stage_name, model, tokenizer, fams, episodes=20, max_steps=18)
    df.to_csv(RUNS_DIR/f'eval_summary_{stage_name}.csv', index=False)
    summaries[stage_name]=summary
    print(stage_name, summary)
baselines=OBS_ROOT/'baselines.json'
payload={'random':{},'scripted':{},'llm_base':{},'trained_llm':{}}
if baselines.exists(): payload=json.loads(baselines.read_text(encoding='utf-8'))
payload['trained_llm']=summaries.get('stageC',{})
baselines.write_text(json.dumps(payload, indent=2), encoding='utf-8')
print('Updated', baselines)
required=[RUNS_DIR/'training_curves_stageA.csv', RUNS_DIR/'training_curves_stageB.csv', RUNS_DIR/'training_curves_stageC.csv', RUNS_DIR/'eval_summary_stageA.csv', RUNS_DIR/'eval_summary_stageB.csv', RUNS_DIR/'eval_summary_stageC.csv', Path('outputs/grpo-stage-b'), Path('outputs/grpo-stage-c')]
print('Artifact check:')
for p in required:
    if p.is_dir():
        files=[x for x in p.rglob('*') if x.is_file()]
        print(p, '->', 'OK' if files else 'EMPTY')
    else:
        print(p, '->', 'OK' if p.exists() else 'MISSING')
